In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.0 MB/s eta 0:00:00


In [3]:
import os
import shutil
import urllib.request
import tarfile
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
yolo_dir = "/content/datasets/food101"
if os.path.exists(yolo_dir):
    shutil.rmtree(yolo_dir)

In [ ]:
url = "http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz"
file_name = "food-101.tar.gz"
if not os.path.exists(file_name):
    print("Downloading Food-101 dataset (~5GB). This will take a few minutes...")
    urllib.request.urlretrieve(url, file_name)
    print("Download complete!")

Download complete!


In [ ]:
if not os.path.exists("food-101"):
    print("Extracting files...")
    with tarfile.open(file_name, "r:gz") as tar:
        tar.extractall()
    print("Extraction complete!")

Extracting files...


/tmp/ipykernel_1458/4169645871.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Extraction complete!


In [ ]:
print("Formatting for YOLO...")
base_dir = "food-101"
images_dir = os.path.join(base_dir, "images")
meta_dir = os.path.join(base_dir, "meta")

os.makedirs(os.path.join(yolo_dir, "train"), exist_ok=True)
os.makedirs(os.path.join(yolo_dir, "val"), exist_ok=True)

def split_data(split_name, txt_file):
    with open(os.path.join(meta_dir, txt_file), 'r') as f:
        lines = f.read().splitlines()

    for line in lines:
        line = line.strip()
        if not line: continue

        class_name, img_name = line.split('/')
        src_img = os.path.join(images_dir, class_name, f"{img_name}.jpg")

        dest_folder = os.path.join(yolo_dir, split_name, class_name)
        os.makedirs(dest_folder, exist_ok=True)
        dest_img = os.path.join(dest_folder, f"{img_name}.jpg")

        # Moving instead of copying to save Colab disk space
        if os.path.exists(src_img):
            shutil.move(src_img, dest_img)

print("Moving training data...")
split_data("train", "train.txt")
print("Moving validation data...")
split_data("val", "test.txt")

Formatting for YOLO...
Moving training data...
Moving validation data...


In [ ]:
shutil.rmtree(base_dir)
os.remove(file_name)

print("Dataset is ready for YOLO! 🎉")

Dataset is ready for YOLO! 🎉


In [ ]:
model = YOLO('yolov8s-cls.pt')

path = '/content/datasets/food101'

results = model.train(
    data= path,
    epochs=20,
    imgsz=224,
    batch=64,
    device=0,
    project='NutritionScanner',
    name='food_model'
)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/food101, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_model2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspect

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train: Scanning /content/datasets/food101/train... 75750 images, 0 corrupt: 100% ━━━━━━━━━━━━ 75750/75750 1.1Kit/s 1:10
train: New cache created: /content/datasets/food101/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 17.4±11.2 MB/s, size: 37.4 KB)
val: Scanning /content/datasets/food101/val... 25250 images, 0 corrupt: 100% ━━━━━━━━━━━━ 25250/25250 966.5it/s 26.1s
val: New cache created: /content/datasets/food101/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/NutritionScanner/food_model2
Starting training for 20 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/20      1.36G       3.62         38        224: 10

KeyboardInterrupt: 

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# drive_path = '/content/drive/MyDrive/NutritionScanner_Runs'
# os.makedirs(drive_path, exist_ok=True)

# print("Copying your saved weights to Google Drive...")
# shutil.copytree('/content/runs/classify/NutritionScanner/food_model2', f'{drive_path}/food_model', dirs_exist_ok=True)
# print("Backup complete!")

Copying your saved weights to Google Drive...
Backup complete! 🚀


In [ ]:
model = YOLO('/content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/last.pt')

results = model.train(resume=True)

Ultralytics 8.4.30 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/food101, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_model2, nbs=64, nms=False, opset=None, optimize=False, 

KeyboardInterrupt: 

In [ ]:
# shutil.copytree('/content/runs/classify/NutritionScanner/food_model2', '/content/drive/MyDrive/NutritionScanner_Runs/food_model', dirs_exist_ok=True)
# print("2nd Backup Complete last")

2nd Backup Complete last


In [10]:
model = YOLO('/content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/last.pt')

results = model.train(resume=True)

Ultralytics 8.4.31 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/food101, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_model2, nbs=64, nms=False, opset=None, optimize=False, 

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


train: Scanning /content/datasets/food101/train... 75750 images, 0 corrupt: 100% ━━━━━━━━━━━━ 75750/75750 836.8it/s 1:31
train: New cache created: /content/datasets/food101/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 20.6±5.7 MB/s, size: 37.4 KB)
val: Scanning /content/datasets/food101/val... 25250 images, 0 corrupt: 100% ━━━━━━━━━━━━ 25250/25250 851.5it/s 29.7s
val: New cache created: /content/datasets/food101/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Resuming training /content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/last.pt from epoch 11 to 20 total epochs
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/NutritionScanner/food_model2
Starting training for 20 epo

In [11]:
shutil.copytree('/content/runs/classify/NutritionScanner/food_model2', '/content/drive/MyDrive/NutritionScanner_Runs/food_model', dirs_exist_ok=True)

'/content/drive/MyDrive/NutritionScanner_Runs/food_model'

In [12]:
model = YOLO('/content/drive/MyDrive/NutritionScanner_Runs/food_model/weights/best.pt')

test_image_url = 'https://www.tasteofhome.com/wp-content/uploads/2018/01/Homemade-Pizza_EXPS_FT23_376_EC_120123_3.jpg'

results = model(test_image_url)

prediction = results[0]
predicted_class = prediction.names[prediction.probs.top1]
confidence = prediction.probs.top1conf.item() * 100

print(f"Predicted Food: {predicted_class.upper()}")
print(f"Confidence: {confidence:.2f}%")


WARNING ⚠️ Download failure, retrying 1/3 https://www.tasteofhome.com/wp-content/uploads/2018/01/Homemade-Pizza_EXPS_FT23_376_EC_120123_3.jpg... HTTP Error 403: Forbidden
image 1/1 /content/Homemade-Pizza_EXPS_FT23_376_EC_120123_3.jpg: 224x224 pizza 1.00, lasagna 0.00, omelette 0.00, french_toast 0.00, apple_pie 0.00, 3.3ms
Speed: 19.1ms preprocess, 3.3ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Predicted Food: PIZZA
Confidence: 99.91%
